# Kalman Forward SHADOW Bake-off V1

V3.4 이후 historical tuning을 중단하고 새로운 데이터에서 세 allocator를 병렬 추적하기 위한 file-only bootstrap입니다.

### Strategies
- A: Hybrid + Equal Weight
- B: Hybrid + Static Max Sharpe
- C: Hybrid + risk_cap_110

### Hybrid sleeves
- US: V2 nested final-refit
- KR: V3 return-regime final-refit
- BTC: V2 nested final-refit

### Execution semantics
- signal history is immutable/deduplicated by timestamp
- virtual sleeves are replayed by the Kalman Native Ledger
- portfolio results are recalculated from historical seed + forward sleeve returns
- no Toss orders, no Neon write, no dashboard visibility, no production write

If all markets do not yet have post-seed realized returns, `WAITING_FOR_FORWARD_DATA` is the correct bootstrap result.


In [ ]:
from google.colab import drive
import json
import shutil
import subprocess
import sys
import traceback
from datetime import datetime
from pathlib import Path

PINNED_SHA = "261f2fae068386bf6424324eba0881d8f43e5734"
SOURCE_BRANCH = "feature/shadow-bakeoff-v1-20260913"

drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive")
MODEL_ROOT = DRIVE_ROOT / "Market_Model_V2"
MATRIX_DIR = MODEL_ROOT / "historical_matrices_v1"
V2_ROOT = MODEL_ROOT / "historical_quant_2017_v2_candidate" / "20260913_nested_v2_001"
V3_ROOT = MODEL_ROOT / "historical_quant_2017_v3_candidate" / "20260913_return_regime_v3_001"
V34_SUMMARY = (
    MODEL_ROOT
    / "historical_quant_2017_v3_4_allocator_gate"
    / "20260913_v3_4_allocator_regime_gate_001"
    / "v3_4_summary.json"
)
OUTPUT_DIR = MODEL_ROOT / "shadow_bakeoff" / "v1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HIST_RAW = (
    DRIVE_ROOT
    / "Market_Data"
    / "v2"
    / "raw"
    / "historical_2017"
    / "multimarket_raw_2017_present.parquet"
)
HIST_FEATURES = (
    DRIVE_ROOT
    / "Market_Features"
    / "v2"
    / "talib"
    / "historical_2017"
    / "multimarket_features_2017_present_v0_3.parquet"
)

STATUS = OUTPUT_DIR / "bootstrap_status.json"
FAILURE = OUTPUT_DIR / "bootstrap_failure.json"
LOG = OUTPUT_DIR / "bootstrap_full_log.txt"
LOG.write_text("", encoding="utf-8")


def run(cmd, *, cwd=None):
    args = [str(x) for x in cmd]
    header = "\n$ " + " ".join(args) + "\n"
    print(header, end="")
    chunks = [header]
    proc = subprocess.Popen(
        args,
        cwd=str(cwd) if cwd else None,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        chunks.append(line)
        print(line, end="")
    rc = proc.wait()
    output = "".join(chunks)
    with LOG.open("a", encoding="utf-8") as fh:
        fh.write(output)
    if rc != 0:
        raise subprocess.CalledProcessError(rc, args, output=output)
    return output


def write_json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2, default=str) + "\n",
        encoding="utf-8",
    )
    tmp.replace(path)


def status(state, phase, **extra):
    write_json(
        STATUS,
        {
            "status": state,
            "phase": phase,
            "updated_at": datetime.now().astimezone().isoformat(),
            "pinned_sha": PINNED_SHA,
            "research_only": True,
            "file_only": True,
            "live_execution": False,
            "toss_execution": False,
            "neon_write": False,
            "production_write": False,
            **extra,
        },
    )


try:
    for required in (
        HIST_RAW,
        HIST_FEATURES,
        V2_ROOT / "historical_v2_candidate_summary.json",
        V3_ROOT / "historical_v3_candidate_summary.json",
        V34_SUMMARY,
    ):
        assert required.exists(), required

    repo = Path("/content/Codex")
    if repo.exists():
        shutil.rmtree(repo)

    status("RUNNING", "CLONE")
    run(
        [
            "git",
            "clone",
            "--branch",
            SOURCE_BRANCH,
            "https://github.com/kimtk94/Codex.git",
            repo,
        ]
    )
    run(["git", "-C", repo, "checkout", "--detach", PINNED_SHA])
    checked = subprocess.check_output(
        ["git", "-C", repo, "rev-parse", "HEAD"],
        text=True,
    ).strip()
    assert checked == PINNED_SHA, (checked, PINNED_SHA)

    app = repo / "kalman-toss-gateway"
    scorer = app / "research" / "shadow_bakeoff" / "forward_scorer.py"
    runner = app / "research" / "shadow_bakeoff" / "runner.py"
    test_file = app / "tests" / "test_shadow_bakeoff_v1.py"
    hist_spec = app / "config" / "model-v2-historical-spec.json"
    v2_spec = app / "config" / "model-v2-historical-candidate-spec.json"
    v3_spec = app / "config" / "model-v3-historical-return-regime-spec.json"

    for required in (scorer, runner, test_file, hist_spec, v2_spec, v3_spec):
        assert required.exists(), required

    status("RUNNING", "ISOLATED_ENV")
    if shutil.which("uv") is None:
        run([sys.executable, "-m", "pip", "install", "-q", "uv"])
    uv = shutil.which("uv")
    assert uv

    venv = Path("/content/.venv-kalman-shadow-bakeoff-v1")
    if venv.exists():
        shutil.rmtree(venv)
    run([uv, "venv", venv])
    vpy = venv / "bin" / "python"

    run(
        [
            uv,
            "pip",
            "install",
            "--python",
            vpy,
            "pandas==3.0.5",
            "numpy==2.4.6",
            "pyarrow",
            "scipy<1.18",
            "scikit-learn",
            "PyPortfolioOpt==1.6.0",
            "pytest",
            "python-dotenv",
        ]
    )
    run([uv, "pip", "check", "--python", vpy])

    versions = run(
        [
            vpy,
            "-c",
            (
                "import pandas as pd,numpy as np,sklearn,pypfopt; "
                "print('pandas='+pd.__version__); "
                "print('numpy='+np.__version__); "
                "print('sklearn='+sklearn.__version__); "
                "print('pypfopt='+pypfopt.__version__)"
            ),
        ],
        cwd=app,
    )

    status("RUNNING", "STATIC_AND_UNIT_TESTS", versions=versions)
    run(
        [vpy, "-m", "py_compile", scorer, runner, test_file],
        cwd=app,
    )
    run(
        [vpy, "-m", "pytest", "-q", "tests/test_shadow_bakeoff_v1.py"],
        cwd=app,
    )

    status("RUNNING", "REFRESH_HISTORICAL_MATRIX")
    run(
        [
            vpy,
            "-m",
            "research.model_v2.build_historical_feature_matrix",
            "--raw-parquet",
            HIST_RAW,
            "--feature-parquet",
            HIST_FEATURES,
            "--spec",
            hist_spec,
            "--output-dir",
            MATRIX_DIR,
            "--start-date",
            "2017-01-01",
        ],
        cwd=app,
    )

    status("RUNNING", "FORWARD_SHADOW_BAKEOFF")
    run(
        [
            vpy,
            "-m",
            "research.shadow_bakeoff.runner",
            "--matrix-dir",
            MATRIX_DIR,
            "--v2-root",
            V2_ROOT,
            "--v3-root",
            V3_ROOT,
            "--v2-spec",
            v2_spec,
            "--v3-spec",
            v3_spec,
            "--v3-4-summary",
            V34_SUMMARY,
            "--output-dir",
            OUTPUT_DIR,
            "--code-sha",
            PINNED_SHA,
        ],
        cwd=app,
    )

    bakeoff_status = OUTPUT_DIR / "latest" / "bakeoff_status.json"
    assert bakeoff_status.exists(), bakeoff_status
    payload = json.loads(bakeoff_status.read_text(encoding="utf-8"))

    status(
        "COMPLETE",
        "DONE",
        tracking_status=payload.get("tracking_status"),
        seed_end=payload.get("seed_end"),
        latest_as_of=payload.get("latest_as_of"),
        post_seed_return_rows=payload.get("post_seed_return_rows"),
        bakeoff_status=str(bakeoff_status),
        versions=versions,
    )

    print("\n" + "=" * 96)
    print("KALMAN FORWARD SHADOW BAKE-OFF V1 BOOTSTRAP COMPLETE")
    print("=" * 96)
    print(bakeoff_status.read_text(encoding="utf-8"))

except Exception as exc:
    payload = {
        "status": "FAIL",
        "updated_at": datetime.now().astimezone().isoformat(),
        "pinned_sha": PINNED_SHA,
        "error_type": type(exc).__name__,
        "error": str(exc),
        "child_output": getattr(exc, "output", None),
        "traceback": traceback.format_exc(),
        "full_log": str(LOG),
        "research_only": True,
        "live_execution": False,
        "toss_execution": False,
        "neon_write": False,
        "production_write": False,
    }
    write_json(FAILURE, payload)
    status(
        "FAIL",
        "FAILED",
        error_type=payload["error_type"],
        error=payload["error"],
        failure_json=str(FAILURE),
        full_log=str(LOG),
    )
    print("\nFAILURE SAVED:", FAILURE)
    print(FAILURE.read_text(encoding="utf-8"))
    raise
